## 04 - Sentiment Data Quality
Cel: ocena kompletności, poprawności i przydatności danych sentymentu Alpha Vantage.
Tabele: bronze.av_sentiment, silver.av_sentiment, gold.av_sentiment_aggregated, gold.av_sentiment_sector_daily, gold.sentiment_vs_returns, gold.sentiment_lead_lag

In [0]:
%sql
SELECT 
  'bronze' AS layer, 
  COUNT(*) AS total_rows 
FROM bronze.av_sentiment

UNION ALL

SELECT 
  'silver' AS layer, 
  COUNT(*) AS total_rows 
FROM silver.av_sentiment

In [0]:
%sql
SELECT 
  symbol, 
  COUNT(*) AS article_count,
  MIN(date) AS min_date,
  MAX(date) AS max_date
FROM silver.av_sentiment
GROUP BY symbol
ORDER BY article_count DESC

In [0]:
%sql
SELECT 
  ticker_sentiment_label, 
  COUNT(*) AS cnt,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM silver.av_sentiment
GROUP BY ticker_sentiment_label
ORDER BY cnt DESC

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT symbol) AS symbols,
  MIN(date) AS min_date,
  MAX(date) AS max_date,
  ROUND(AVG(article_count), 2) AS avg_daily_articles,
  ROUND(AVG(avg_sentiment_score), 4) AS avg_sentiment
FROM gold.av_sentiment_aggregated

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT symbol) AS symbols
FROM gold.sentiment_vs_returns

In [0]:
%sql
SELECT DISTINCT symbol 
FROM gold.av_sentiment_aggregated
WHERE symbol NOT IN (
  SELECT DISTINCT symbol 
  FROM gold.sentiment_vs_returns
)

In [0]:
%sql
DESCRIBE gold.sentiment_lead_lag

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT symbol) AS symbols,
  COUNT(return_t1) AS has_1d,
  COUNT(return_t2) AS has_3d,
  COUNT(return_t5) AS has_5d
FROM gold.sentiment_lead_lag

In [0]:
%sql
SELECT DISTINCT symbol
FROM gold.sentiment_vs_returns
WHERE symbol NOT IN (
  SELECT DISTINCT symbol
  FROM gold.sentiment_lead_lag
)

In [0]:
%sql
SELECT 
  sector, 
  COUNT(*) AS total_days,
  COUNT(DISTINCT date) AS unique_days,
  ROUND(AVG(article_count), 2) AS avg_daily_articles
FROM gold.av_sentiment_sector_daily
GROUP BY sector
ORDER BY total_days DESC

In [0]:
%sql
SELECT 
  industry, 
  COUNT(*) AS total_days,
  ROUND(AVG(article_count), 2) AS avg_daily_articles
FROM gold.av_sentiment_sector_daily
GROUP BY industry
ORDER BY total_days DESC

### Wnioski
1. Bronze 3737 → Silver 3110. 627 rekordów (17%) usuniętych jako duplikaty. AV API zwraca powtarzające się artykuły, pipeline obsługuje to poprawnie.
2. 41 symboli z sentymentem - pełne pokrycie technology. Ale jakość nierówna: NVDA 387 artykułów, ARM/DDOG 3. 10+ symboli ma <10 artykułów. Zakresy dat dramatycznie różne (2015-2026).
3. Rozkład sentymentu: 38% neutral, 53% pozytywny, 8% negatywny. Wystarczający rozrzut na analizę. Bearish tylko 53 rekordy (1.7%) - za mało na wnioski o silnie negatywnym sentymencie.
4. Gold sentiment aggregated: 893 wierszy, 41 symboli, średnio 22 dni per symbol, 2.64 artykuły dziennie. Lekko pozytywny bias (avg score 0.18).
5. Sentiment vs returns: 535 wierszy, 40 symboli. 40% straty z agregacji - weekendy/święta. ZS wypadł (4 artykuły).
6. Sentiment lead lag: 477 wierszy, 39 symboli. ARM wypadł (3 artykuły na końcu szeregu). Zero nulli w forward returns.
7. Sentyment sektorowy tylko dla technology (wynika z filtru w pipeline). 7 industry w ramach tech. Semiconductors i software-application solidna próbka (100+ dni). Communication equipment (16), consumer electronics (12) - za mało na trendy.

Lecimy z `05_gold_readiness`?